### Step 1: Install Dependencies

In [1]:
! pip install torch transformers datasets accelerate seqeval

> You **MUST** work with GPU, so check first:

In [2]:
import torch

if torch.cuda.is_available():
    print("GPU is available!")
else:
    print("Please switch to a GPU-enabled environment.")

GPU is available!


### Step 2: Import Libraries

In [3]:
from collections import Counter
! pip install datasets
# ! pip install load_datase
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
from seqeval.metrics import precision_score, recall_score, f1_score
from torch.quantization import quantize_dynamic


### Step 3: Dataset Preparation

#### 3.1: Load Data

In [4]:
import pandas as pd

In [5]:
# Load "lhoestq/conll2003" dataset using HuggingFace datasets library
dataset = load_dataset("lhoestq/conll2003")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


#### 3.2: Show Data

In [6]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


In [7]:
print(dataset["train"].features["ner_tags"].feature)

Value('int64')


**Seperate train , test , validate data into seperate datsaets and changing them to pandas**

In [8]:
# dataset["train"].to_pandas().to_csv("hugging_face_train.csv" , index = False)
# train = pd.read_csv("hugging_face_train.csv")

In [9]:
# dataset['validation'].to_pandas().to_csv("hugging_face_validate.csv"  , index = False)
# validate = pd.read_csv("hugging_face_validate.csv")

In [10]:
# dataset["test"].to_pandas().to_csv("hugging_face_test.csv" , index = False)
# test = pd.read_csv("hugging_face_test.csv")

In [11]:
# # Show samples of data before start
# pd.set_option('display.max_colwidth', None)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)

# print(train.head(1))

tokens : diving the sentence into words

pos_tags : part of speech

chunk_tags : identifies grammatical phrases

ner_tags : named entity recognition , recognizes the person , organization etc ..

**to check the data balance here , we'll need to map the different classes to numbers**

In [12]:
label_names = [
    "O",
    "B-PER", "I-PER",
    "B-ORG", "I-ORG",
    "B-LOC", "I-LOC",
    "B-MISC", "I-MISC"
]

In [13]:
all_tags = []
for sample in dataset["train"]:
    all_tags.extend(sample["ner_tags"])

# All labels from all sentences

counter = Counter(all_tags)

label_counts = {}

for idx, count in counter.most_common():  #converts numbers of the index to names
    print(label_names[idx], ":", count)
    label_counts[label_names[idx]] = count


O : 169578
B-LOC : 7140
B-PER : 6600
B-ORG : 6321
I-PER : 4528
I-ORG : 3704
B-MISC : 3438
I-LOC : 1157
I-MISC : 1155


O  -->  not an entity

PER  -->  Person

ORG  -->  Organization

LOC  -->  Location

MISC  -->  Miscellaneous
ex : nationalities , events , adjectives

B -- > Beginning of the entity

I --> Inside the entity
ex: New York



#3.3: Data Formatting
> Convert dataset into BERT-compatible format
<br>FROM:<br>
{'id': '0',
 'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}
 <br><br>
 TO:<br>
 {'id': '0',
 'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0],
 'input_ids': [101,
  7270,
  22961,
  1528,
  1840,
  1106,
  21423,
  1418,
  2495,
  12913,
  119,
  102],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'labels': [-100, 3, 0, 7, 0, 0, 0, 7, 0, 0, 0, -100]}

Where:
- input_ids will be the X and labels will be the Y
- 101 token id is the <sos> token
- 102 token id is the <eos> token
- -100 is the label for <sos> and <eos>

# Expected Input (X)
input_ids → numbers (BERT tokens)

attention_mask → which tokens
matter

token_type_ids → sentence
segment (usually 0)

# Expected Output (Y)
labels → NER tags aligned with BERT tokens

# Tokenize

In [14]:
# use the following tokenizer "bert-base-cased"

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [15]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    all_labels = []

    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_id = None
        label_ids = []

        for word_id in word_ids:

            if word_id is None:
                # special tokens like [CLS], [SEP]
                label_ids.append(-100)

            elif word_id != previous_word_id:
                # first subword of a word → keep label
                label_ids.append(labels[word_id])

            else:
                # subword of same word → ignore
                label_ids.append(-100)

            previous_word_id = word_id

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

In [16]:
tokenized_dataset = dataset.map(tokenize_and_align_labels , batched = True)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

# Step 4: Model Loading

In [17]:
num_labels = len(label_names)

id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

In [18]:
# Load pre-trained BERT model with a token classification head
from transformers import BertForTokenClassification

model = BertForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=9,
    id2label={i: l for i, l in enumerate(label_names)},
    label2id={l: i for i, l in enumerate(label_names)}
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

In [19]:
# Check the model architecture
print(model.config)

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "O",
    "1": "B-PER",
    "2": "I-PER",
    "3": "B-ORG",
    "4": "I-ORG",
    "5": "B-LOC",
    "6": "I-LOC",
    "7": "B-MISC",
    "8": "I-MISC"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "B-LOC": 5,
    "B-MISC": 7,
    "B-ORG": 3,
    "B-PER": 1,
    "I-LOC": 6,
    "I-MISC": 8,
    "I-ORG": 4,
    "I-PER": 2,
    "O": 0
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,


### Step 5: Fine-Tuning

#### 5.1: Hyperparameter Selection

In [20]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=500,
    logging_dir="./logs",
    logging_steps=10,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


#### 5.2: Evaluation Metrices

In [21]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    true_labels = [
        [id2label[l] for l in label if l != -100]
        for label in labels
    ]

    true_preds = [
        [id2label[p] for p, l in zip(pred, label) if l != -100]
        for pred, label in zip(preds, labels)
    ]

    precision = precision_score(true_labels, true_preds)
    recall = recall_score(true_labels, true_preds)
    f1 = f1_score(true_labels, true_preds)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

#### 5.3: Training Loop
> ONLY Choose one of the following methods NOT BOTH

In [22]:
from transformers import DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer)

In [25]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Default Standard Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics  # Custom function to calculate F1, precision, recall
)

trainer.train()

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [26]:
sample = tokenized_dataset["train"][0]

print("labels:", sample["labels"])
print("min:", min([x for x in sample["labels"] if x != -100]))
print("max:", max(sample["labels"]))

labels: [-100, 3, 0, 7, 0, 0, 0, 7, 0, 0, -100]
min: 0
max: 7


It is a Hugging Face training engine that handles:

forward pass (prediction)

loss computation

backpropagation

optimizer updates

evaluation

logging

saving checkpoints

In [ ]:
# OR Custom Trainer
! pip install AdamW
from torch.utils.data import DataLoader
from transformers import AdamW

train_dataloader = DataLoader(tokenized_dataset["train"],
                              batch_size=16,
                              shuffle=True)
optimizer = AdamW(model.parameters(),
                  lr=2e-5)

for epoch in range(3):  # Number of epochs
    model.train()
    for batch in train_dataloader:
        inputs = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**inputs)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

### Step 6: Save Weights

In [ ]:
# Save the model and tokenizer
model.save_pretrained("./ner_model")
tokenizer.save_pretrained("./ner_model")

print("Model and tokenizer saved to ./ner_model")

### Step 7: Predict & Test

In [ ]:
def predict(text: str):
    tokens = tokenizer(text,
                       return_tensors="pt",
                       truncation=True,
                       is_split_into_words=True)
    with torch.no_grad():
        outputs = model(**tokens)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1).squeeze().tolist()
    return {"tokens": tokenizer.tokenize(text), "predictions": predictions}

### Step 8: Model Quantization

In [ ]:
quantized_model = quantize_dynamic(
    model, {torch.nn.Linear}, dtype=torch.qint8
)
quantized_model.save_pretrained("./quantized_ner_model")
print("Quantized model saved.")

In [ ]:
def predict(text: str):
    tokens = tokenizer(text,
                       return_tensors="pt",
                       truncation=True,
                       is_split_into_words=True)
    with torch.no_grad():
        outputs = quantized_model(**tokens)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1).squeeze().tolist()
    return {"tokens": tokenizer.tokenize(text), "predictions": predictions}

### Step 9: Comparison and Conclusion

In [ ]:
# Compare the trained & quantized models' predictions and give a comment
# Compare the time taken for both models using time library and give a comment


> Thanls a lot for your Effort